# ResNet18-2.5D vs STU-Net-S: tres fusiones trimodales

Compara **concatenación proyectada**, **fusión convexa** y **atención cruzada** usando tabular + texto + visión. ResNet y STU-Net se restringen a la intersección exacta de pacientes y usan los mismos cinco hold-outs 80/20 estratificados por evento.

El notebook solo configura y orquesta. La extracción tabular, el control de leakage, PCA/Cox, las fusiones y el resumen pareado se importan o ejecutan desde `clinical_core`. Los embeddings visuales deben existir previamente en Drive.


In [ ]:
# 1. Montar Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Configuración editable (rutas compatibles con los notebooks de ResNet)
from pathlib import Path

REPO_URL = 'https://github.com/raulprtech/clinical_core.git' # @param {type:"string"}
REPO_REF = 'master' # @param {type:"string"}
REPO_PATH = Path('/content/clinical_core')
DRIVE_ROOT = Path('/content/drive/MyDrive/datatesis')

XML_DATA_DIR = DRIVE_ROOT / 'data/raw/clinicalsupplement'
OUTCOMES_CSV = DRIVE_ROOT / 'data/raw/tcga_kirc_os_official_444.csv'
TEXT_EMBEDDINGS = DRIVE_ROOT / 'data/embeddings/text_embeddings_TCGA-KIRC_20260528.npz'
RESNET_EMBEDDINGS = DRIVE_ROOT / 'baseline_resnet18_2p5d_exp6_official444/artifacts/vision_embedding_patient_level_all_usable.csv'
# Ajusta esta ruta al CSV de 160 pacientes exportado por tu experimento STU-Net.
STUNET_EMBEDDINGS = DRIVE_ROOT / 'data/embeddings/vision/stunet_fp32_160/stunet_s_fp32_embeddings_768.csv'

CACHE_DIR = DRIVE_ROOT / 'cache/trimodal_fusion_inputs'
FEATURES_CSV = CACHE_DIR / 'raw_features.csv'
RUN_NAME = 'resnet18_2p5d_vs_stunet_s_three_fusions_official444_v1' # @param {type:"string"}
OUTPUT_DIR = DRIVE_ROOT / 'results' / RUN_NAME
SEEDS = [42, 123, 456, 789, 1024]
MAX_EPOCHS = 300 # @param {type:"integer"}
PATIENCE = 30 # @param {type:"integer"}
RESUME = True # @param {type:"boolean"}

print('Salida persistente:', OUTPUT_DIR)


In [ ]:
# 3. Instalar/actualizar clinical_core y las dependencias de esta evaluación
import subprocess, sys

if not (REPO_PATH / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO_PATH)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_PATH), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_PATH), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_PATH), 'pull', '--ff-only', 'origin', REPO_REF], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy', 'pandas', 'scikit-learn', 'lifelines', 'pyyaml'
], check=True)

required_tools = [
    REPO_PATH / 'code/tools/run_paired_vision_fusion_benchmark.py',
    REPO_PATH / 'code/tools/evaluate_trimodal_fusion.py',
    REPO_PATH / 'code/tools/evaluate_cross_attention_fusion.py',
]
missing_tools = [str(path) for path in required_tools if not path.is_file()]
if missing_tools:
    raise RuntimeError(
        'El REPO_REF seleccionado todavía no contiene los evaluadores requeridos: ' + str(missing_tools)
    )
print('clinical_core listo en', REPO_PATH)


In [ ]:
# 4. Preparar la tabla clínica reutilizando el extractor del repositorio
import sys
sys.path.insert(0, str(REPO_PATH / 'code'))

CACHE_DIR.mkdir(parents=True, exist_ok=True)
if FEATURES_CSV.is_file():
    print('Reutilizando:', FEATURES_CSV)
else:
    if not XML_DATA_DIR.is_dir():
        raise FileNotFoundError(f'No existe el directorio XML: {XML_DATA_DIR}')
    from components.adapters.ingestion.tabular.utils.extractor import TCGAExtractor
    mapping = REPO_PATH / 'code/components/adapters/ingestion/tabular/configs/tabular_mapping.yaml'
    extractor = TCGAExtractor(str(mapping))
    features, extracted_targets = extractor.extract_cohort(str(XML_DATA_DIR))
    features.to_csv(FEATURES_CSV)
    extracted_targets.to_csv(CACHE_DIR / 'raw_targets_from_xml_audit.csv')
    print('Tabla clínica creada:', FEATURES_CSV, features.shape)

# El desenlace oficial444 se conserva como fuente única para ambos encoders.
if not OUTCOMES_CSV.is_file():
    raise FileNotFoundError(f'Falta el desenlace oficial: {OUTCOMES_CSV}')


In [ ]:
# 5. Preflight: verificar entradas antes de gastar tiempo de cómputo
inputs = {
    'tabular': FEATURES_CSV,
    'outcomes': OUTCOMES_CSV,
    'text': TEXT_EMBEDDINGS,
    'resnet18_2p5d': RESNET_EMBEDDINGS,
    'stunet_s_frozen': STUNET_EMBEDDINGS,
}
missing = {name: str(path) for name, path in inputs.items() if not path.is_file()}
for name, path in inputs.items():
    state = 'OK' if path.is_file() else 'FALTA'
    size = f'{path.stat().st_size / 1024**2:.1f} MB' if path.is_file() else ''
    print(f'{state:5} {name:18} {size:>10}  {path}')
if missing:
    raise FileNotFoundError('Corrige las rutas marcadas como FALTA: ' + str(missing))


## Protocolo que ejecutará la siguiente celda

1. Calcula la intersección **tabular ∩ texto ∩ ResNet ∩ STU-Net ∩ desenlace válido**.
2. Fuerza esa misma cohorte en ambos encoders.
3. Reutiliza los cinco seeds y los mismos hold-outs 80/20 estratificados.
4. Ajusta imputación, escalado, PCA, Cox, pesos convexos y época de redes exclusivamente con datos de train.
5. Guarda resultados por seed, pesos, atención y diferencias pareadas STU-Net − ResNet en Drive.


In [ ]:
# 6. Ejecutar las seis combinaciones (2 encoders x 3 fusiones)
command = [
    sys.executable, '-u',
    str(REPO_PATH / 'code/tools/run_paired_vision_fusion_benchmark.py'),
    '--features', str(FEATURES_CSV),
    '--targets', str(OUTCOMES_CSV),
    '--text-embeddings', str(TEXT_EMBEDDINGS),
    '--resnet-embeddings', str(RESNET_EMBEDDINGS),
    '--stunet-embeddings', str(STUNET_EMBEDDINGS),
    '--output-dir', str(OUTPUT_DIR),
    '--seeds', *map(str, SEEDS),
    '--max-epochs', str(MAX_EPOCHS),
    '--patience', str(PATIENCE),
]
if RESUME:
    command.append('--resume')

print('Ejecutando:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
# 7. Tabla final y deltas pareados
import json
import pandas as pd
from IPython.display import display

fusion_results = pd.read_csv(OUTPUT_DIR / 'fusion_results_summary.csv')
encoder_deltas = pd.read_csv(OUTPUT_DIR / 'encoder_paired_summary.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())

display(fusion_results.sort_values('cindex_mean', ascending=False).style.format({
    'cindex_mean': '{:.3f}',
    'cindex_std_across_seeds': '{:.3f}',
    'delta_vs_vision_mean': '{:+.3f}',
    'delta_vs_best_unimodal_per_seed_mean': '{:+.3f}',
}))
display(encoder_deltas.style.format({
    'delta_stunet_minus_resnet_mean': '{:+.3f}',
    'delta_std_across_seeds': '{:.3f}',
}))
print('Mejor configuración por media:', summary['best_mean_configuration'])
print('Artefactos guardados en:', OUTPUT_DIR)


## Lectura del resultado

Una fusión aporta valor si su `delta_vs_vision_mean` es positivo de forma consistente, no solo si gana en una seed. Para comparar encoders dentro de una fusión, usa `encoder_paired_summary.csv`: un valor positivo de `delta_stunet_minus_resnet_mean` favorece STU-Net. Conserva también los resultados por seed; cinco hold-outs permiten una comparación pareada, pero no deben interpretarse como cinco cohortes independientes.
